# Push-T × slots — la recette émergente face à DINO-WM (SR 0.90)

Workflow : **1** (code+deps) → **2** (Drive) → puis selon le besoin :
- **3** collecte de données (une fois, déjà fait si pusht_data.npz est sur Drive)
- **4** entraînement / reprise (le checkpoint pusht_wm.pt vit sur Drive, architecture auto-lue)
- **5** diagnostics sans réentraîner  |  **6** figures

État (2026-07-06) : perception 0.096 ✓ ; fix imagine() séquentiel poussé → **cellule 4B (reprise)** est la prochaine étape.

In [ ]:
# 1) Code + dépendances
import os
if not os.path.exists('/content/jepa-physics-bench'):
    !git clone https://github.com/frpatry/jepa-physics-bench.git /content/jepa-physics-bench
!cd /content/jepa-physics-bench && git pull
%cd /content/jepa-physics-bench
!pip install --quiet gym-pusht 'pymunk<7'
print('OK')

In [ ]:
# 2) Drive (données + checkpoints persistants)
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception as e:
    print('pas de Drive:', e)

In [ ]:
# 3) COLLECTE (une fois — sauter si pusht_data.npz est déjà sur Drive)
#!python -u pusht_data.py --n 10000 --T 6

In [ ]:
# 4A) ENTRAÎNEMENT depuis zéro (config gagnante du run 6)
#!python -u slots_pusht.py --steps 40000 --bs 32 --slot_dim 12 --sig 0.05

In [ ]:
# 4B) REPRISE du checkpoint (architecture auto-lue) — PROCHAINE ÉTAPE (run 7 : recalibrer
#     les alphas après le fix imagine() séquentiel)
!python -u slots_pusht.py --steps 30000 --bs 32 --sig 0.05 --resume 1

In [ ]:
# 5) DIAGNOSTICS sans réentraîner : pipeline (peel vs imagine vs +1 pas) + masques par prise
!python -u slots_pusht.py --load 1 --diag 1

In [ ]:
# 6) Figures
from IPython.display import Image, display
for f in ['/content/slots_pusht.png', '/content/slots_pusht_diag.png', '/content/pusht_data.png']:
    if os.path.exists(f): display(Image(f))